# 🧰 Python en IA — Partie 3 · La boîte à outils

**L'idée :** Python doit sa place en IA à ses **librairies**, des boîtes à outils gratuites écrites par des milliers de développeurs. Aujourd'hui, on en ouvre quatre, plus une en bonus, toujours sur le même fichier : les commandes d'*Amora Zone*.

| Module | Librairie | Son rôle | La question business |
|---|---|---|---|
| M8 | **pandas** | préparer et résumer les données | Quand et où nos clients achètent-ils ? |
| M9 | **NumPy** | calculer vite sur des tableaux de nombres | Une campagne -20 % est-elle rentable ? |
| M10 | **Plotly** | des graphiques interactifs | Comment montrer nos ventes à la direction ? |
| M11 | **Folium** | des cartes interactives | Où ouvrir la prochaine boutique ? |
| M12+ | **Prophet** *(bonus)* | prévoir l'avenir d'une série temporelle | Combien vendra-t-on cet hiver ? |

### Le même plan pour chaque librairie
1. 🪪 **Carte d'identité** : à quoi elle sert, d'où elle vient, son lien avec l'IA ;
2. 🧱 **Les basiques** : les quelques lignes pour démarrer ;
3. ✨ **Sa spécificité** : ce qu'elle fait mieux que les autres ;
4. ✏️ **Les exercices** : sur un cas Amora Zone.

### 📖 Le fichier `commandes.csv` (le même qu'en Partie 2)
Une ligne = une commande : `id_commande`, `id_client`, `date_commande` (date et heure), `canal` (boutique ou site web), `ville` (de la boutique ou de livraison), `latitude`, `longitude`, `categorie`, `nb_articles`, `code_promo`, `montant`.

### 🧭 Mode d'emploi
1. **Fichier → Enregistrer une copie dans Drive.**
2. Binôme : 🧑‍✈️ **Pilote** = tape le code · 🧭 **Copilote** = lit les consignes et guide. **On inverse à chaque module.**
3. Exécute les cellules **dans l'ordre**, avec **Maj + Entrée**.
4. 🔮 = prédis le résultat avant d'exécuter · `A_COMPLETER` = à compléter · 💡 = indice · 🚀 = pour aller plus loin · 🆘 = cellule de secours (la solution).
5. 🔁 Chaque module commence par une cellule **CHECKPOINT** qui recharge tout ce dont il a besoin : chaque module peut se faire seul.

# M8 · pandas, le chef d'orchestre
👥 **On inverse les rôles !**

### 🪪 Carte d'identité
| | |
|---|---|
| **Sert à** | lire, nettoyer, filtrer, regrouper et résumer des tableaux de données |
| **Né en** | 2008, créé par Wes McKinney, un analyste financier lassé des limites d'Excel |
| **On l'importe avec** | `import pandas as pd` |
| **Lien avec l'IA** | avant d'entraîner un modèle, il faut préparer les données. C'est souvent **la majorité du travail** d'un projet d'IA, et c'est le métier de pandas |

Tu as déjà vu les bases en Partie 2 (`read_csv`, `head`, filtres, `groupby`). Ce module montre **deux spécificités** très utilisées en entreprise : **le temps** et **les tableaux croisés**.

In [ ]:
# 🔁 CHECKPOINT — recrée les données dont ce module a besoin
# Ce module réutilise : le fichier commandes, chargé et avec des dates converties
# → Tout s'est bien passé jusqu'ici ? L'exécuter ne change rien, mais ça ne casse rien non plus.
# → Tu as eu une erreur, sauté une section ou redémarré la session ? Exécute-la avant de continuer.

import pandas as pd
import numpy as np

URL_DATA = "https://raw.githubusercontent.com/SAMIMZOUGHI/MBADMB_python/main/data/"
commandes = pd.read_csv(URL_DATA + "commandes.csv")
commandes["date_commande"] = pd.to_datetime(commandes["date_commande"])
print("✅ commandes est en mémoire :", len(commandes), "commandes")

### 🧱 Les basiques, en 30 secondes

In [ ]:
print(commandes.shape)        # (lignes, colonnes)
commandes.head(3)

### ✨ Spécificité 1 — Découper le temps avec `.dt`
Une colonne de dates contient beaucoup d'informations cachées : le mois, le jour de la semaine, l'heure… On y accède avec **`.dt`**, suivi de ce qu'on veut extraire.

| Code | Donne | Exemple |
|---|---|---|
| `.dt.year` | l'année | 2025 |
| `.dt.month` | le mois (1 à 12) | 12 |
| `.dt.day_name()` | le nom du jour | Saturday |
| `.dt.hour` | l'heure | 20 |

In [ ]:
commandes["heure"] = commandes["date_commande"].dt.hour
commandes["jour"] = commandes["date_commande"].dt.day_name()

commandes[["date_commande", "heure", "jour"]].head()

### ✨ Spécificité 2 — Changer d'échelle de temps avec `resample`
`resample` est un **`groupby` pour les dates** : il fait des paquets par jour, par semaine ou par mois.

Il a besoin que les dates soient l'**index** du tableau (l'étiquette des lignes, M5). On le fait avec `.set_index("date_commande")`.

| Code | Paquets |
|---|---|
| `"D"` | par jour |
| `"W"` | par semaine |
| `"ME"` | par mois (*Month End* : chaque paquet est étiqueté par le dernier jour du mois) |

In [ ]:
ca_mensuel = commandes.set_index("date_commande").resample("ME")["montant"].sum()

ca_mensuel.round(0)       # 🔮 quel mois est le plus fort ? le plus faible ?

👉 **Lecture :** le pic de **décembre** (Noël), le creux d'**août** (vacances), une hausse en **mai** (fête des mères). Et d'une année sur l'autre, les ventes **augmentent**. On retrouvera cette courbe avec Plotly (M10) et Prophet (M12+).

### ✏️ Exercice 8.1 — Quand nos clients achètent-ils ?
1. Trouve le **mois record** de chiffre d'affaires et son montant.
2. Compte le **nombre de commandes par heure**, séparément pour la boutique et pour le site web.

💡 *Indice :* 1) `.idxmax()` donne l'**étiquette** de la plus grande valeur (ici, la date), `.max()` sa valeur. 2) Filtre d'abord un canal (M4), puis `value_counts()` sur la colonne `heure`, et `.sort_index()` pour ranger les heures dans l'ordre.
🚀 *Bonus :* le jour de la semaine où l'on vend le plus.

In [ ]:
mois_record = A_COMPLETER   # ✏️ À COMPLÉTER
print("Mois record :", mois_record, "→", round(ca_mensuel.max()), "€")

site = commandes[commandes["canal"] == "site web"]
print(site["heure"].A_COMPLETER.sort_index())   # ✏️ À COMPLÉTER

boutique = A_COMPLETER   # ✏️ À COMPLÉTER
print(boutique["heure"].value_counts().sort_index())

In [ ]:
# @title 🆘 Cellule de secours — exercice 8.1 (la solution : à n'ouvrir qu'après avoir essayé)
mois_record = ca_mensuel.idxmax()
print("Mois record :", mois_record, "→", round(ca_mensuel.max()), "€")
print(commandes[commandes["canal"] == "site web"]["heure"].value_counts().sort_index())
print(commandes[commandes["canal"] == "boutique"]["heure"].value_counts().sort_index())

👉 **Lecture business :** en boutique, les commandes suivent les heures d'ouverture (10 h – 19 h). Sur le site, le **pic est le soir, vers 20 h**. C'est le bon moment pour envoyer un email ou une notification.

### ✨ Spécificité 3 — Le tableau croisé dynamique : `pivot_table`
`pivot_table` construit un **tableau croisé**, comme dans Excel : des lignes, des colonnes, des valeurs et un calcul.

In [ ]:
croise = commandes.pivot_table(
    index="ville",          # en lignes : les villes
    columns="canal",        # en colonnes : boutique / site web
    values="montant",       # la valeur à calculer
    aggfunc="sum",          # le calcul : la somme
    fill_value=0,           # 0 au lieu d'une case vide quand il n'y a rien
)
croise.round(0)

### ✏️ Exercice 8.2 — Les villes sans boutique
1. Ajoute à `croise` une colonne `total` (boutique + site web).
2. Trie le tableau par `total`, du plus grand au plus petit.
3. Garde seulement les villes **sans boutique** (CA en boutique = 0) et affiche-les.

💡 *Indice :* 1) `croise["boutique"] + croise["site web"]` · 2) `.sort_values("total", ascending=False)` · 3) un filtre (M4) sur `croise["boutique"] == 0`.
🚀 *Bonus :* la part du site web dans le CA de chaque ville, en %.

In [ ]:
croise["total"] = A_COMPLETER   # ✏️ À COMPLÉTER
croise = croise.sort_values(A_COMPLETER, ascending=False)   # ✏️ À COMPLÉTER

sans_boutique = croise[A_COMPLETER]   # ✏️ À COMPLÉTER
sans_boutique.round(0)

In [ ]:
# @title 🆘 Cellule de secours — exercice 8.2 (la solution : à n'ouvrir qu'après avoir essayé)
croise["total"] = croise["boutique"] + croise["site web"]
croise = croise.sort_values("total", ascending=False)
sans_boutique = croise[croise["boutique"] == 0]
sans_boutique.round(0)

👉 **Une découverte :** **Rennes** et **Montpellier** n'ont pas de boutique Amora Zone, mais leurs clients achètent en ligne presque autant qu'à **Lille**, qui en a une. On le verra sur une carte en M11.

### 🎯 Ce que pandas prépare pour la suite
Chaque librairie suivante a besoin d'un tableau **déjà résumé** : la courbe mensuelle pour Plotly, un tableau par ville pour Folium. Le réflexe : **pandas prépare, les autres librairies affichent ou calculent.**

In [ ]:
par_ville = commandes.groupby("ville").agg(
    ca=("montant", "sum"),
    nb_commandes=("id_commande", "count"),
    latitude=("latitude", "first"),          # "first" : la première valeur (identique pour toute la ville)
    longitude=("longitude", "first"),
    nb_en_boutique=("canal", lambda c: (c == "boutique").sum()),
)
par_ville["a_une_boutique"] = par_ville["nb_en_boutique"] > 0

par_ville.sort_values("ca", ascending=False).round(1)

👉 La ligne `lambda c: (c == "boutique").sum()` est une **mini-fonction sans nom** : pour chaque ville, elle compte les commandes passées en boutique (les `True` valent 1, M4). Pas besoin de la retenir, mais sache la reconnaître : on en croise souvent.

# M9 · NumPy, le moteur caché
👥 **On inverse les rôles !**

### 🪪 Carte d'identité
| | |
|---|---|
| **Sert à** | calculer très vite sur des tableaux de nombres (des **arrays**) |
| **Né en** | 2006, pour le calcul scientifique |
| **On l'importe avec** | `import numpy as np` |
| **Lien avec l'IA** | **le socle de tout le reste.** pandas est construit dessus, et les modèles d'IA (scikit-learn, PyTorch, TensorFlow) manipulent des tableaux de nombres de la même façon : une image, un son, un texte deviennent des tableaux NumPy |

Tu l'utilises depuis le début sans le savoir : chaque colonne pandas est un tableau NumPy déguisé.

In [ ]:
# 🔁 CHECKPOINT — recrée les données dont ce module a besoin
# Ce module réutilise : le fichier commandes
# → Tout s'est bien passé jusqu'ici ? L'exécuter ne change rien, mais ça ne casse rien non plus.
# → Tu as eu une erreur, sauté une section ou redémarré la session ? Exécute-la avant de continuer.

import pandas as pd
import numpy as np

URL_DATA = "https://raw.githubusercontent.com/SAMIMZOUGHI/MBADMB_python/main/data/"
commandes = pd.read_csv(URL_DATA + "commandes.csv")
commandes["date_commande"] = pd.to_datetime(commandes["date_commande"])
print("✅ commandes est en mémoire :", len(commandes), "commandes")

### 🧱 Les basiques
Un **array** ressemble à une liste (M2.5), avec une différence capitale : **une opération s'applique à tous les éléments d'un coup**.

In [ ]:
prix = np.array([24.90, 18.00, 45.00, 9.50])

print(prix * 2)              # 🔮 avec une liste, [24.9, 18.0] * 2 répète la liste ; avec un array ?
print(prix.mean())           # la moyenne
print(prix.max(), prix.sum())

👉 `prix * 2` multiplie **chaque** prix par 2. Avec une liste Python, `* 2` aurait **répété** la liste (M2.2 : le même symbole, des résultats différents selon le type).

On récupère la colonne `montant` sous forme d'array avec `.to_numpy()` :

In [ ]:
montants = commandes["montant"].to_numpy()

print(type(montants))
print(montants.shape)        # 🔮 combien d'éléments ?
print(montants[:5])          # les 5 premiers (même syntaxe que les listes)

### ✨ Spécificité 1 — La vectorisation : la vitesse
**Question :** appliquer une remise de 15 % à **1 million** de prix. Deux façons :
- **une boucle `for`** (M2.8) : Python traite les prix **un par un** ;
- **NumPy** : une seule opération sur tout le tableau. On appelle ça **vectoriser**.

🔮 Laquelle est la plus rapide, et de combien ?

In [ ]:
import time

un_million = np.resize(montants, 1_000_000)      # on répète nos montants pour atteindre 1 million

# 1. Avec une boucle
debut = time.perf_counter()
remises_boucle = []
for m in un_million:
    remises_boucle.append(m * 0.85)
duree_boucle = time.perf_counter() - debut

# 2. Avec NumPy
debut = time.perf_counter()
remises_numpy = un_million * 0.85
duree_numpy = time.perf_counter() - debut

print(f"Boucle : {duree_boucle:.3f} s")
print(f"NumPy  : {duree_numpy:.4f} s")
print(f"NumPy est {duree_boucle / duree_numpy:.0f} fois plus rapide")

👉 Même résultat, mais NumPy est **des dizaines de fois plus rapide** (environ 50 fois sur Colab ; le chiffre varie d'une machine à l'autre). Sur 1 million de lignes, la différence se compte en fractions de seconde ; sur les milliards de calculs d'un modèle d'IA, c'est la différence entre une heure et un mois. **C'est pour ça que l'IA est écrite avec des tableaux, pas avec des boucles.**

`time.perf_counter()` lit un chronomètre : on le lit avant et après, la différence donne la durée. `{duree_boucle:.3f}` affiche le nombre avec 3 décimales.

### ✏️ Exercice 9.1 — Les statistiques d'un tableau
Avec l'array `montants`, calcule :
1. le **CA total** et le **CA après une remise de 10 %** sur toutes les commandes ;
2. la **médiane** et les **quartiles** (25 % et 75 %) des montants ;
3. la **corrélation** entre le nombre d'articles et le montant.

💡 *Indice :* 2) `np.percentile(montants, [25, 50, 75])` (la médiane = 50 %) · 3) `np.corrcoef(a, b)` renvoie un petit tableau 2 × 2 ; la corrélation est en position `[0, 1]`.
🚀 *Bonus :* combien de commandes dépassent 150 € ? Un `True` vaut 1 (M4) : `(montants > 150).sum()`.

In [ ]:
ca = A_COMPLETER   # ✏️ À COMPLÉTER
ca_remise = A_COMPLETER   # ✏️ À COMPLÉTER
print("CA :", round(ca), "€ | après -10 % :", round(ca_remise), "€")

quartiles = A_COMPLETER   # ✏️ À COMPLÉTER
print("Quartiles 25 %, 50 %, 75 % :", quartiles.round(2))

articles = commandes["nb_articles"].to_numpy()
correlation = A_COMPLETER   # ✏️ À COMPLÉTER
print("Corrélation articles / montant :", round(correlation, 2))

In [ ]:
# @title 🆘 Cellule de secours — exercice 9.1 (la solution : à n'ouvrir qu'après avoir essayé)
ca = montants.sum()
ca_remise = (montants * 0.90).sum()
print("CA :", round(ca), "€ | après -10 % :", round(ca_remise), "€")
print("Quartiles :", np.percentile(montants, [25, 50, 75]).round(2))
articles = commandes["nb_articles"].to_numpy()
print("Corrélation :", round(np.corrcoef(articles, montants)[0, 1], 2))

👉 **Comment lire la corrélation :** elle va de **-1** (quand l'un monte, l'autre descend) à **+1** (ils montent ensemble). **0,63** = un lien net : plus d'articles, panier plus élevé. Mais pas parfait, car un coffret cadeau à 45 € pèse plus qu'un ingrédient à 9 €.

### ✨ Spécificité 2 — Décider sur tout le tableau : `np.where`
`np.where(condition, si_vrai, si_faux)` est un **`if / else` (M2.7) appliqué à chaque élément**, sans boucle.

In [ ]:
livraison = np.where(montants >= 50, 0, 4.90)      # offerte dès 50 €, sinon 4,90 €

print(livraison[:8])
print("Frais de livraison facturés au total :", round(livraison.sum()), "€")

### ✏️ Exercice 9.2 — Petits, moyens et gros paniers
Classe chaque commande : **« petit »** sous 40 €, **« moyen »** de 40 à 100 €, **« gros »** au-delà. Puis compte les commandes de chaque taille.

💡 *Indice :* deux `np.where` imbriqués, comme un `if / elif / else` : `np.where(montants < 40, "petit", np.where(montants < 100, "moyen", "gros"))`. Pour compter : `np.unique(taille, return_counts=True)`.
🚀 *Bonus :* ajoute ce résultat comme nouvelle colonne `taille_panier` dans `commandes`, puis calcule le CA par taille avec un `groupby`.

In [ ]:
taille = np.where(montants < 40, "petit", np.where(A_COMPLETER, "moyen", A_COMPLETER))   # ✏️ À COMPLÉTER

valeurs, nombres = np.unique(taille, return_counts=True)
print(valeurs)
print(nombres)

In [ ]:
# @title 🆘 Cellule de secours — exercice 9.2 (la solution : à n'ouvrir qu'après avoir essayé)
taille = np.where(montants < 40, "petit", np.where(montants < 100, "moyen", "gros"))
print(np.unique(taille, return_counts=True))

### ✨ Spécificité 3 — Simuler le hasard : la méthode de Monte-Carlo
**La question de la directrice marketing :** « Si on envoie un code **-20 %** à tous nos clients, gagne-t-on de l'argent ? »

On ne connaît pas l'avenir, mais on a des **fourchettes** :
- entre **15 % et 35 %** des commandes utiliseront le code (on perd 20 % sur celles-là) ;
- la campagne fera vendre entre **2 % et 12 %** de plus.

**L'idée de Monte-Carlo :** au lieu de deviner une seule réponse, on **tire au hasard 10 000 scénarios** dans ces fourchettes et on regarde combien sont gagnants. NumPy le fait en **une ligne par étape**, sans boucle.

In [ ]:
rng = np.random.default_rng(42)          # un générateur de hasard ; 42 = tout le monde obtient les mêmes tirages
n = 10_000                               # nombre de scénarios

taux_utilisation = rng.uniform(0.15, 0.35, n)     # 10 000 tirages entre 15 % et 35 %
hausse_ventes = rng.uniform(0.02, 0.12, n)        # 10 000 tirages entre 2 % et 12 %

ca_actuel = montants.sum()
ca_campagne = ca_actuel * (1 + hausse_ventes) * (1 - taux_utilisation * 0.20)

print("CA actuel :", round(ca_actuel), "€")
print("CA moyen avec la campagne :", round(ca_campagne.mean()), "€")
print(ca_campagne[:5].round(0))          # les 5 premiers scénarios

**Comment se lit le calcul :** pour chaque scénario, le CA augmente de la hausse des ventes (`1 + hausse`), puis on retire la remise de 20 % sur la part des commandes qui utilisent le code (`1 - taux × 0,20`). Les 10 000 scénarios sont calculés **en même temps**.

### ✏️ Exercice 9.3 — La campagne est-elle rentable ?
1. Calcule la **part des scénarios** où le CA avec campagne dépasse le CA actuel.
2. Calcule l'**intervalle à 90 %** : les percentiles 5 % et 95 % de `ca_campagne`.
3. Refais la simulation avec une remise de **-30 %** au lieu de -20 %. Que devient la part des scénarios gagnants ?

💡 *Indice :* 1) `(ca_campagne > ca_actuel)` donne 10 000 `True` / `False` ; `.mean()` donne la part de `True` · 2) `np.percentile(ca_campagne, [5, 95])` · 3) recopie la ligne `ca_campagne = ...` en remplaçant `0.20` par `0.30`.
🚀 *Bonus :* à partir de quelle remise la campagne devient-elle perdante dans plus de la moitié des scénarios ?

In [ ]:
part_gagnante = A_COMPLETER   # ✏️ À COMPLÉTER
print("Scénarios gagnants :", round(part_gagnante * 100), "%")

intervalle = A_COMPLETER   # ✏️ À COMPLÉTER
print("Dans 90 % des cas, le CA sera entre", intervalle.round(0), "€")

ca_campagne_30 = A_COMPLETER   # ✏️ À COMPLÉTER
print("Avec -30 % :", round((ca_campagne_30 > ca_actuel).mean() * 100), "% de scénarios gagnants")

In [ ]:
# @title 🆘 Cellule de secours — exercice 9.3 (la solution : à n'ouvrir qu'après avoir essayé)
part_gagnante = (ca_campagne > ca_actuel).mean()
print("Scénarios gagnants :", round(part_gagnante * 100), "%")
print("Intervalle à 90 % :", np.percentile(ca_campagne, [5, 95]).round(0))
ca_campagne_30 = ca_actuel * (1 + hausse_ventes) * (1 - taux_utilisation * 0.30)
print("Avec -30 % :", round((ca_campagne_30 > ca_actuel).mean() * 100), "%")

👉 **La réponse à la directrice :** « Avec -20 %, la campagne est gagnante dans environ **2 cas sur 3**. Avec -30 %, elle est perdante **6 fois sur 10**. La bascule se situe vers -27 %. » C'est une réponse bien plus utile qu'un chiffre unique, parce qu'elle dit **à quel point on est sûr**. Les modèles d'IA raisonnent souvent ainsi : en **probabilités**.

# M10 · Plotly, des graphiques qui se manipulent
👥 **On inverse les rôles !**

### 🪪 Carte d'identité
| | |
|---|---|
| **Sert à** | des graphiques **interactifs** : survol, zoom, filtres par clic sur la légende |
| **Né en** | 2012, par une start-up de Montréal ; la librairie Python est gratuite |
| **On l'importe avec** | `import plotly.express as px` (le module le plus simple) |
| **Lien avec l'IA** | c'est l'outil des dashboards de suivi : performances d'un modèle, résultats d'une segmentation, prévisions |

**La différence avec matplotlib (Partie 2) :** matplotlib fait une **image** ; Plotly fait une **page interactive**. Survole un point, zoome, double-clique pour revenir.

In [ ]:
# 🔁 CHECKPOINT — recrée les données dont ce module a besoin
# Ce module réutilise : commandes, ca_mensuel et par_ville (préparés en M8)
# → Tout s'est bien passé jusqu'ici ? L'exécuter ne change rien, mais ça ne casse rien non plus.
# → Tu as eu une erreur, sauté une section ou redémarré la session ? Exécute-la avant de continuer.

import pandas as pd
import numpy as np

URL_DATA = "https://raw.githubusercontent.com/SAMIMZOUGHI/MBADMB_python/main/data/"
commandes = pd.read_csv(URL_DATA + "commandes.csv")
commandes["date_commande"] = pd.to_datetime(commandes["date_commande"])
print("✅ commandes est en mémoire :", len(commandes), "commandes")

ca_mensuel = commandes.set_index("date_commande").resample("ME")["montant"].sum()
par_ville = commandes.groupby("ville").agg(
    ca=("montant", "sum"),
    nb_commandes=("id_commande", "count"),
    latitude=("latitude", "first"),
    longitude=("longitude", "first"),
    nb_en_boutique=("canal", lambda c: (c == "boutique").sum()),
)
par_ville["a_une_boutique"] = par_ville["nb_en_boutique"] > 0
print("✅ ca_mensuel et par_ville sont en mémoire")

### 🧱 Les basiques : une ligne par graphique
Avec `plotly.express`, un graphique s'écrit toujours de la même façon :

`px.TYPE(tableau, x="colonne", y="colonne", ...)`

| Type | Graphique |
|---|---|
| `px.bar` | barres |
| `px.line` | courbe |
| `px.scatter` | nuage de points |
| `px.histogram` | histogramme |
| `px.pie` | camembert |

In [ ]:
import plotly.express as px

ca_canal = commandes.groupby("canal")["montant"].sum().reset_index()   # reset_index : l'index redevient une colonne
fig = px.bar(ca_canal, x="canal", y="montant", title="CA par canal (€)")
fig.show()

🖱️ **Passe la souris sur une barre** : la valeur exacte s'affiche. En haut à droite, des boutons pour zoomer ou télécharger l'image.

👉 `reset_index()` : après un `groupby`, le canal est passé en index. Plotly a besoin de **colonnes** : `reset_index()` le remet en colonne.

### ✨ Spécificité 1 — La courbe interactive

In [ ]:
mensuel = ca_mensuel.reset_index()          # deux colonnes : date_commande et montant

fig = px.line(mensuel, x="date_commande", y="montant", markers=True,
              title="CA mensuel d'Amora Zone (€)")
fig.show()

🖱️ **Zoome** en traçant un rectangle sur la période de Noël, puis **double-clique** pour revenir. Le pic de décembre et le creux d'août sautent aux yeux.

### ✨ Spécificité 2 — Une couleur par catégorie : `color=`
Ajouter `color="colonne"` crée **une couleur par valeur** de cette colonne, avec une légende **cliquable** : un clic masque une série, un double-clic l'isole.

In [ ]:
commandes["mois"] = commandes["date_commande"].dt.to_period("M").astype(str)    # "2025-12"
ca_mois_cat = commandes.groupby(["mois", "categorie"])["montant"].sum().reset_index()

fig = px.bar(ca_mois_cat, x="mois", y="montant", color="categorie",
             title="CA mensuel par catégorie (€)")
fig.show()

🖱️ **Double-clique sur « Coffret cadeau » dans la légende** pour l'isoler. 🔮 Qu'observes-tu en novembre et décembre ?

👉 Les **coffrets cadeaux** représentent environ **un tiers** des ventes en novembre-décembre, contre environ 1 sur 10 le reste de l'année. Une info précieuse pour préparer les stocks.

### ✏️ Exercice 10.1 — Le CA par ville
Trace un graphique en barres du **CA par ville**, trié du plus grand au plus petit, avec une couleur selon que la ville **a une boutique ou non**.

💡 *Indice :* pars de `par_ville` (préparé en M8), trie-le par `ca`, puis `reset_index()`. Dans `px.bar` : `x="ville"`, `y="ca"`, `color="a_une_boutique"`.
🚀 *Bonus :* ajoute `text_auto=".2s"` dans `px.bar` pour afficher les montants sur les barres (`.2s` = 2 chiffres significatifs, « 25k »).

In [ ]:
tableau = par_ville.sort_values(A_COMPLETER, ascending=False).reset_index()   # ✏️ À COMPLÉTER

fig = px.bar(tableau, x=A_COMPLETER, y=A_COMPLETER, color=A_COMPLETER,   # ✏️ À COMPLÉTER
             title="CA par ville (€) — avec ou sans boutique")
fig.show()

In [ ]:
# @title 🆘 Cellule de secours — exercice 10.1 (la solution : à n'ouvrir qu'après avoir essayé)
tableau = par_ville.sort_values("ca", ascending=False).reset_index()
fig = px.bar(tableau, x="ville", y="ca", color="a_une_boutique", title="CA par ville (€)")
fig.show()

### ✏️ Exercice 10.2 — La carte de chaleur des commandes en ligne
Quand les clients commandent-ils sur le site ? Trace une **carte de chaleur** : le jour de la semaine en colonnes, l'heure en lignes, la couleur = le nombre de commandes.

💡 *Indice :* `px.density_heatmap(tableau, x="colonne", y="colonne")` compte les lignes pour chaque case. Filtre d'abord le site web. Il te faut les colonnes `jour` et `heure` : elles sont créées dans la cellule ci-dessous.
🚀 *Bonus :* fais la même carte pour la boutique et compare.

In [ ]:
commandes["heure"] = commandes["date_commande"].dt.hour
commandes["jour"] = commandes["date_commande"].dt.day_name()
ordre_jours = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

In [ ]:
site = commandes[A_COMPLETER]   # ✏️ À COMPLÉTER

fig = px.density_heatmap(site, x=A_COMPLETER, y=A_COMPLETER, category_orders={"jour": ordre_jours},   # ✏️ À COMPLÉTER
                         title="Commandes en ligne : jour × heure")
fig.show()

In [ ]:
# @title 🆘 Cellule de secours — exercice 10.2 (la solution : à n'ouvrir qu'après avoir essayé)
site = commandes[commandes["canal"] == "site web"]
fig = px.density_heatmap(site, x="jour", y="heure", category_orders={"jour": ordre_jours})
fig.show()

### ✨ Spécificité 3 — Partager un graphique interactif
`fig.write_html("fichier.html")` enregistre le graphique dans une **page web** autonome. Ton manager l'ouvre dans son navigateur, sans Python, et garde le survol et le zoom.

In [ ]:
fig = px.line(ca_mensuel.reset_index(), x="date_commande", y="montant", markers=True,
              title="CA mensuel d'Amora Zone (€)")
fig.write_html("ca_mensuel.html")
print("💾 ca_mensuel.html créé : dossier 📁 à gauche, clic droit → Télécharger")

# M11 · Folium, des cartes interactives
👥 **On inverse les rôles !**

### 🪪 Carte d'identité
| | |
|---|---|
| **Sert à** | des **cartes interactives** (zoom, déplacement, infobulles), comme Google Maps |
| **Repose sur** | Leaflet, une librairie de cartes web très utilisée ; les fonds de carte viennent d'OpenStreetMap |
| **On l'importe avec** | `import folium` |
| **Lien avec l'IA** | le **géomarketing** : où implanter un magasin, cibler une zone, optimiser des livraisons. Les modèles de prévision utilisent souvent des données géographiques |

**La question business :** Amora Zone veut ouvrir **une nouvelle boutique**. Où ?

In [ ]:
# 🔁 CHECKPOINT — recrée les données dont ce module a besoin
# Ce module réutilise : commandes et par_ville (préparés en M8)
# → Tout s'est bien passé jusqu'ici ? L'exécuter ne change rien, mais ça ne casse rien non plus.
# → Tu as eu une erreur, sauté une section ou redémarré la session ? Exécute-la avant de continuer.

import pandas as pd
import numpy as np

URL_DATA = "https://raw.githubusercontent.com/SAMIMZOUGHI/MBADMB_python/main/data/"
commandes = pd.read_csv(URL_DATA + "commandes.csv")
commandes["date_commande"] = pd.to_datetime(commandes["date_commande"])
print("✅ commandes est en mémoire :", len(commandes), "commandes")

ca_mensuel = commandes.set_index("date_commande").resample("ME")["montant"].sum()
par_ville = commandes.groupby("ville").agg(
    ca=("montant", "sum"),
    nb_commandes=("id_commande", "count"),
    latitude=("latitude", "first"),
    longitude=("longitude", "first"),
    nb_en_boutique=("canal", lambda c: (c == "boutique").sum()),
)
par_ville["a_une_boutique"] = par_ville["nb_en_boutique"] > 0
print("✅ ca_mensuel et par_ville sont en mémoire")

### 🧱 Les basiques : une carte et un marqueur
Une carte se crée avec un **centre** (latitude, longitude) et un niveau de **zoom** (plus le chiffre est grand, plus on est près). Puis on y **ajoute** des éléments avec `.add_to(carte)`.

In [ ]:
import folium

carte = folium.Map(location=[46.6, 2.4], zoom_start=6)      # le centre de la France, vu de loin

folium.Marker(
    location=[48.8566, 2.3522],                             # [latitude, longitude] de Paris
    popup="Boutique Amora Zone · Paris",                    # le texte au clic
    tooltip="Paris",                                        # le texte au survol
).add_to(carte)

carte          # la dernière ligne affiche la carte

🖱️ Zoome, déplace la carte, **clique sur le marqueur**.

👉 La **latitude** dit à quelle hauteur on est (nord / sud), la **longitude** à quelle distance du méridien de Greenwich (est / ouest). Le fichier contient les deux pour chaque ville.

### ✨ Spécificité 1 — Un marqueur par ville, avec une boucle
On parcourt le tableau `par_ville` ligne par ligne. `.iterrows()` donne, à chaque tour, le **nom de la ligne** (la ville, qui est l'index) et **la ligne elle-même**, qu'on lit comme un dictionnaire (M2.6).

In [ ]:
carte = folium.Map(location=[46.6, 2.4], zoom_start=6)

for ville, ligne in par_ville.iterrows():                   # un tour par ville
    folium.Marker(
        location=[ligne["latitude"], ligne["longitude"]],
        tooltip=f"{ville} : {ligne['ca']:,.0f} €",           # :,.0f = séparateur de milliers, sans décimale
    ).add_to(carte)

carte

### ✨ Spécificité 2 — Des cercles proportionnels
Un marqueur dit **où**. Un **cercle** peut aussi dire **combien** (sa taille) et **quoi** (sa couleur). C'est `folium.CircleMarker`, avec :

| Paramètre | Rôle |
|---|---|
| `radius=` | la taille du cercle, en pixels |
| `color=` / `fill_color=` | la couleur du contour / de l'intérieur |
| `fill=True`, `fill_opacity=0.6` | remplir le cercle, avec un peu de transparence |
| `popup=` | le texte au clic |

### ✏️ Exercice 11.1 — La carte du chiffre d'affaires
Dessine un cercle par ville :
- **taille** proportionnelle au CA : `radius = ca / 2000` ;
- **couleur** : **vert** si la ville a une boutique, **rouge** sinon ;
- **popup** : le nom de la ville et son CA.

💡 *Indice :* reprends la boucle ci-dessus et remplace `folium.Marker` par `folium.CircleMarker`. Pour la couleur, une condition sur une ligne : `couleur = "green" if ligne["a_une_boutique"] else "red"`.
🚀 *Bonus :* ajoute au popup le nombre de commandes.

In [ ]:
carte = folium.Map(location=[46.6, 2.4], zoom_start=6)

for ville, ligne in par_ville.iterrows():
    couleur = A_COMPLETER   # ✏️ À COMPLÉTER
    folium.CircleMarker(
        location=[ligne["latitude"], ligne["longitude"]],
        radius=A_COMPLETER,   # ✏️ À COMPLÉTER
        color=couleur,
        fill=True,
        fill_color=couleur,
        fill_opacity=0.6,
        popup=f"{ville} : {ligne['ca']:,.0f} €",
    ).add_to(carte)

carte

In [ ]:
# @title 🆘 Cellule de secours — exercice 11.1 (la solution : à n'ouvrir qu'après avoir essayé)
carte = folium.Map(location=[46.6, 2.4], zoom_start=6)
for ville, ligne in par_ville.iterrows():
    couleur = "green" if ligne["a_une_boutique"] else "red"
    folium.CircleMarker(location=[ligne["latitude"], ligne["longitude"]], radius=ligne["ca"] / 2000,
                        color=couleur, fill=True, fill_color=couleur, fill_opacity=0.6,
                        popup=f"{ville} : {ligne['ca']:,.0f} €").add_to(carte)
carte

👉 **Ce que montre la carte :** deux **gros cercles rouges**, **Rennes** et **Montpellier**. Beaucoup de clients, aucune boutique. C'est exactement ce que le tableau de M8 disait, mais la carte le rend **évident en 2 secondes**, et elle montre en plus que ces villes ne sont proches d'aucune boutique existante.

### ✏️ Exercice 11.2 — La recommandation
Avec `par_ville`, affiche les **3 villes sans boutique** qui ont le plus gros CA, puis écris ta recommandation dans la variable `recommandation`.

💡 *Indice :* un filtre sur `par_ville["a_une_boutique"] == False`, puis `.sort_values("ca", ascending=False).head(3)`.
🚀 *Bonus :* quel critère ajouterais-tu avant de décider (population, loyers, concurrence…) ? Note-le dans la recommandation.

In [ ]:
candidates = par_ville[A_COMPLETER].sort_values("ca", ascending=False).head(3)   # ✏️ À COMPLÉTER
print(candidates[["ca", "nb_commandes"]].round(0))

recommandation = "A_COMPLETER"   # ✏️ À COMPLÉTER
print(recommandation)

In [ ]:
# @title 🆘 Cellule de secours — exercice 11.2 (la solution : à n'ouvrir qu'après avoir essayé)
candidates = par_ville[par_ville["a_une_boutique"] == False].sort_values("ca", ascending=False).head(3)
print(candidates[["ca", "nb_commandes"]].round(0))

### ✨ Spécificité 3 — La carte de chaleur et l'export
`HeatMap` (dans `folium.plugins`) colore les zones selon une intensité : ici, le CA de chaque ville. Et `carte.save("carte.html")` crée une page web à partager, comme avec Plotly.

In [ ]:
from folium.plugins import HeatMap

carte_chaleur = folium.Map(location=[46.6, 2.4], zoom_start=6, tiles="CartoDB positron")   # un fond de carte plus sobre
points = par_ville[["latitude", "longitude", "ca"]].values.tolist()   # une liste [latitude, longitude, poids] par ville
HeatMap(points, radius=35).add_to(carte_chaleur)

carte_chaleur.save("carte_amora_zone.html")
carte_chaleur

# [OPTIONNEL] M12+ · Prophet, prévoir les ventes
👥 **On inverse les rôles !**

### 🪪 Carte d'identité
| | |
|---|---|
| **Sert à** | **prévoir** une série temporelle : ventes, trafic, demande… |
| **Né en** | 2017, rendu public par Meta (Facebook), qui l'utilisait en interne |
| **On l'importe avec** | `from prophet import Prophet` |
| **Lien avec l'IA** | c'est un **modèle** : il apprend sur le passé et produit une prévision avec une **marge d'incertitude** |

**Comment il pense :** il découpe une série en morceaux simples qu'il additionne :
- une **tendance** (les ventes montent ou baissent sur le long terme) ;
- des **saisonnalités** (le cycle de l'année, celui de la semaine) ;
- du **bruit** (ce qu'il ne sait pas expliquer).

**La question :** combien Amora Zone vendra-t-elle dans les 3 prochains mois, Noël compris ?

In [ ]:
# 🔁 CHECKPOINT — recrée les données dont ce module a besoin
# Ce module réutilise : le fichier commandes
# → Tout s'est bien passé jusqu'ici ? L'exécuter ne change rien, mais ça ne casse rien non plus.
# → Tu as eu une erreur, sauté une section ou redémarré la session ? Exécute-la avant de continuer.

import pandas as pd
import numpy as np

URL_DATA = "https://raw.githubusercontent.com/SAMIMZOUGHI/MBADMB_python/main/data/"
commandes = pd.read_csv(URL_DATA + "commandes.csv")
commandes["date_commande"] = pd.to_datetime(commandes["date_commande"])
print("✅ commandes est en mémoire :", len(commandes), "commandes")

### 🧱 Étape 1 — Préparer les données au format Prophet
Prophet impose **un format strict** : un tableau à **deux colonnes**, nommées exactement :
- **`ds`** : la date ;
- **`y`** : la valeur à prévoir.

On calcule le CA **par jour** (`resample("D")`, M8), puis on renomme les colonnes.

In [ ]:
ventes = commandes.set_index("date_commande").resample("D")["montant"].sum().reset_index()
ventes.columns = ["ds", "y"]        # les deux noms imposés par Prophet

print(len(ventes), "jours")
ventes.tail()

### 🧱 Étape 2 — Entraîner et prévoir, en 4 lignes
| Ligne | Rôle |
|---|---|
| `modele = Prophet(yearly_seasonality=True)` | crée le modèle, en lui demandant d'apprendre le **cycle de l'année** |
| `modele.fit(ventes)` | **l'entraîne** : il apprend la tendance et les saisonnalités sur les 2 ans |
| `modele.make_future_dataframe(periods=90)` | prépare les dates à prévoir : les 2 ans connus + 90 jours |
| `modele.predict(futur)` | **prévoit** chaque jour |

L'entraînement prend quelques secondes. Des messages techniques peuvent s'afficher : c'est normal.

⚠️ **Le piège de `yearly_seasonality=True` :** par défaut, Prophet n'apprend le cycle de l'année **que s'il a au moins 2 ans d'historique**. Ici on a 729 jours : un de moins. Sans cette option, il ignorerait Noël et prévoirait une ligne presque plate. C'est pour ça qu'il affiche un avertissement (« less than 730 days ») : on lui force la main en connaissance de cause. **Toujours lire les avertissements d'un modèle.**

In [ ]:
from prophet import Prophet

modele = Prophet(yearly_seasonality=True)     # le cycle annuel, même avec un peu moins de 2 ans d'historique
modele.fit(ventes)

futur = modele.make_future_dataframe(periods=90)     # + 90 jours, jusqu'à fin janvier 2027
prevision = modele.predict(futur)

prevision[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail()

| Colonne | Signification |
|---|---|
| `yhat` | la prévision (le *ŷ*, « y chapeau », des statisticiens) |
| `yhat_lower` / `yhat_upper` | la fourchette : dans 80 % des cas, la vraie valeur tombera entre les deux |

### ✨ Spécificité — Voir la prévision et la décomposer

In [ ]:
import matplotlib.pyplot as plt

modele.plot(prevision, xlabel="Date", ylabel="CA par jour (€)")
plt.title("Prévision du CA journalier (zone bleue = fourchette)")
plt.show()

👉 **Les points noirs** = les ventes réelles. **La ligne bleue** = ce que le modèle a appris, prolongée de 90 jours. **La zone bleu clair** = la fourchette d'incertitude, plus large dans le futur.

In [ ]:
modele.plot_components(prevision)       # la tendance, le cycle de la semaine, le cycle de l'année
plt.show()

👉 Prophet a **retrouvé seul** ce qu'on a vu en M8 : une **tendance** à la hausse, un **cycle annuel** avec le pic de fin d'année et le creux d'août, et un **cycle hebdomadaire** avec plus de ventes le week-end.

### ✏️ Exercice 12.1 — Le chiffre à donner à la direction
1. Garde seulement les **jours futurs** de `prevision` (après la dernière date connue).
2. Calcule le **CA total prévu** sur ces 90 jours, avec sa fourchette basse et haute.
3. Compare avec le CA réel de la **même période un an plus tôt** (du 31 octobre 2025 au 28 janvier 2026).

💡 *Indice :* 1) `prevision[prevision["ds"] > ventes["ds"].max()]` · 2) `.sum()` sur `yhat`, `yhat_lower`, `yhat_upper` · 3) filtre `ventes` entre deux dates : `(ventes["ds"] >= "2025-10-31") & (ventes["ds"] <= "2026-01-28")` (deux conditions : `&`, chacune entre parenthèses).
🚀 *Bonus :* en quel mois des 3 prochains le CA sera-t-il le plus fort ?

In [ ]:
futur_seul = A_COMPLETER   # ✏️ À COMPLÉTER

prevu = futur_seul["yhat"].sum()
bas = A_COMPLETER   # ✏️ À COMPLÉTER
haut = A_COMPLETER   # ✏️ À COMPLÉTER
print(f"CA prévu sur 90 jours : {prevu:,.0f} € (entre {bas:,.0f} et {haut:,.0f} €)")

an_dernier = ventes[A_COMPLETER]["y"].sum()   # ✏️ À COMPLÉTER
print(f"Même période l'an dernier : {an_dernier:,.0f} €")
print(f"Évolution prévue : {(prevu / an_dernier - 1) * 100:+.0f} %")

In [ ]:
# @title 🆘 Cellule de secours — exercice 12.1 (la solution : à n'ouvrir qu'après avoir essayé)
futur_seul = prevision[prevision["ds"] > ventes["ds"].max()]
print("CA prévu :", round(futur_seul["yhat"].sum()), "€ (entre", round(futur_seul["yhat_lower"].sum()),
      "et", round(futur_seul["yhat_upper"].sum()), "€)")

👉 **Lecture :** environ **79 000 €** prévus sur les 3 prochains mois, soit environ **+25 %** par rapport à l'an dernier. C'est cohérent avec la croissance vue en M8 (environ +15 % par an), un peu au-dessus : les très bonnes ventes d'octobre 2026 tirent la tendance vers le haut. Un analyste le signalerait : **la prévision est probablement un peu optimiste.**

⚠️ **Attention à la fourchette additionnée :** sommer les bornes basses jour par jour donne une fourchette **trop large** (il est peu probable que tous les jours soient mauvais en même temps). C'est un ordre de grandeur, pas un calcul rigoureux.

👉 **Ce qu'une prévision n'est pas :** une certitude. Prophet suppose que le futur ressemble au passé. Une crise, un nouveau concurrent, une campagne exceptionnelle : il ne peut pas les deviner. **Le modèle prévoit, le marketeur décide.**

# 🏁 Ta boîte à outils

| Librairie | Ce que tu sais faire | Les lignes à retenir |
|---|---|---|
| **pandas** | découper le temps, croiser, résumer | `.dt.hour`, `resample("ME")`, `pivot_table`, `groupby().agg()` |
| **NumPy** | calculer vite, décider sur tout un tableau, simuler | `np.array`, `np.where`, `np.percentile`, `rng.uniform` |
| **Plotly** | des graphiques interactifs à partager | `px.bar`, `px.line`, `color=`, `px.density_heatmap`, `write_html` |
| **Folium** | des cartes interactives | `folium.Map`, `CircleMarker`, `.add_to(carte)`, `HeatMap` |
| **Prophet** | prévoir une série temporelle | colonnes `ds` / `y`, `fit`, `make_future_dataframe`, `predict` |

### 🧭 Comment choisir une librairie, seul, en stage
1. **Décris le besoin en une phrase** : « je veux une carte », « je veux prévoir ».
2. **Cherche** « python + besoin » : les librairies les plus citées sont souvent les bonnes.
3. **Regarde la documentation** : la page *Getting started* et ses exemples.
4. **Teste sur 10 lignes** avant de l'appliquer à tout le fichier.

### 📚 Les documentations officielles
- pandas : pandas.pydata.org · NumPy : numpy.org
- Plotly : plotly.com/python · Folium : python-visualization.github.io/folium
- Prophet : facebook.github.io/prophet